<div style="background:linear-gradient(120deg,#152642,#245e83);padding:34px 38px;border-radius:16px;color:white;">
<p style="font-size:15px;letter-spacing:2px;margin:0;color:#c5e8ff">PYTHON FOR DATA ANALYSIS · LESSON 043</p>
<h1 style="font-size:36px;margin:12px 0;color:white">See the Story in the Data</h1>


**Today’s workflow:** Ask a question → choose a chart → build it → improve it → explain what it shows.

## Table of contents

1. [Learning goals and chart chooser](#goals)
2. [Setup and dataset](#setup)
3. [Plotting fundamentals: a dot, labels, title, legend, markers](#fundamentals)
4. [Customization and annotation](#custom)
5. [Side-by-side plots and three libraries](#libraries)
6. [Histograms and bin choices](#histograms)
7. [Scatter plots and relationships](#scatter)
8. [Boxplots, violins, and group comparisons](#groups)
9. [Count plots and readable categories](#counts)
10. [Four research investigations: figures → results → observations](#research)
11. [Pitfalls, reproducibility, and exporting](#quality)
12. [In-class challenge and answer guide](#challenge)

> **Instructor suggestion:** Teach Sections 1–5 live, then let students work in pairs on the research investigations. All notebook cells are intended to run from top to bottom.

<a id="goals"></a>
## 1. Learning goals — what makes a good graph?

By the end, students should be able to **(1)** select an appropriate chart, **(2)** set the x-axis, y-axis, title, and legend, **(3)** add a dot, reference line, and annotation, **(4)** arrange two charts side by side, **(5)** create basic plots with Pandas, Matplotlib, and Seaborn, and **(6)** write a cautious, evidence-based observation.

| Research question | Variable types | Best starting chart |
|---|---|---|
| How are values distributed? | One numeric | Histogram |
| Do two numeric measurements move together? | Two numeric | Scatter plot |
| How does a numeric measure vary by category? | Numeric + category | Boxplot / violin |
| How many observations are in each category? | One categorical | Count plot |
| How do two views compare? | Any | Two subplots |

**Think–pair–share:** Why is a histogram of the category `Gender` not appropriate? Why should we avoid joining unrelated people’s scatterplot points with a line?

<a id="setup"></a>
## 2. Setup and data checks

We reuse the **Cartwheel dataset** introduced in the original lesson 043. It contains measurements such as `Wingspan`, `CWDistance`, `Height`, `Gender`, and `Score`. It is a small classroom dataset, so do not generalize its patterns to a population.

The public CSV requires internet access when first loaded in Colab. If the link fails, upload `Cartwheeldata.csv` and change `DATA_URL` to the uploaded file path.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 14,
                     "axes.labelsize": 11, "legend.frameon": False,
                     "savefig.bbox": "tight"})

DATA_URL = "https://raw.githubusercontent.com/tec03/Datasets/main/datasets/Cartwheeldata.csv"
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
# Inspect data quality before plotting — reuse skills from earlier classes
required = ["Wingspan", "CWDistance", "Height", "Gender", "Score"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}; found {df.columns.tolist()}")

print("Missing values in required fields:")
display(df[required].isna().sum().to_frame("missing"))
print("Observed Gender categories:", df["Gender"].dropna().unique().tolist())

# Keep the original df intact. Copy only the columns we need for analysis.
viz = df.copy()
for column in ["Wingspan", "CWDistance", "Height", "Score"]:
    viz[column] = pd.to_numeric(viz[column], errors="coerce")
viz["Gender"] = viz["Gender"].astype("string").str.strip()
print("Ready for plotting. Rows:", len(viz))

**Teaching point:** `pd.to_numeric(..., errors="coerce")` converts invalid numeric strings to missing values. We have not removed missing rows globally: each plot or calculation will use the columns it actually needs. **Keep `df` unchanged** so this lesson never silently edits the original dataset.

<a id="fundamentals"></a>
## 3. Plotting fundamentals: start with one dot

A plot is a **figure** containing one or more **axes** (plot areas). `fig` is the whole canvas; `ax` is where we draw. `ax.set_xlabel(...)` and `ax.set_ylabel(...)` label the axes; `ax.set_title(...)` labels the chart.

A point such as `(2, 3)` means **x = 2, y = 3**. The `marker` parameter controls its shape. `s` in `scatter` is marker *area* (points²).

In [ ]:
# FIRST GRAPH: a single dot with every essential label
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(2, 3, marker="o", s=140, color="royalblue", label="Observation A")
ax.set_title("My first data visualization", fontweight="bold", loc="left")
ax.set_xlabel("X coordinate")
ax.set_ylabel("Y coordinate")
ax.set_xlim(0, 4)
ax.set_ylim(0, 5)
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### The essentials (try changing each one)

| Goal | Matplotlib command | Tip |
|---|---|---|
| Title | `ax.set_title("...")` | State the message or question |
| X-axis label | `ax.set_xlabel("...")` | Include units when known |
| Y-axis label | `ax.set_ylabel("...")` | Label the measurement, not just “Y” |
| Dot | `ax.scatter(2, 3, s=100)` | Or `ax.plot(2, 3, "o")` |
| Color | `color="teal"` | Prefer meaningful, consistent colors |
| Legend | `ax.legend()` | Requires labeled series |
| X/Y limits | `ax.set_xlim(...)` / `ax.set_ylim(...)` | Do not misleadingly truncate bar charts |
| Grid | `ax.grid(alpha=.2)` | Keep it subtle |
| Save | `fig.savefig("figure.png", dpi=200)` | Export after plotting |

**Mini-exercise:** Move the dot to `(3, 4)`, make it a red square (`marker="s"`), and rename the axes.

In [ ]:
# Lines vs markers: choose an encoding deliberately
x_demo = np.arange(1, 6)
y_demo = np.array([3, 5, 4, 7, 6])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(x_demo, y_demo, marker="o", linewidth=2, color="teal")
axes[0].set_title("Ordered observations: line + dots")
axes[1].scatter(x_demo, y_demo, marker="D", s=85, color="darkorange")
axes[1].set_title("Independent observations: dots")
for ax in axes:
    ax.set_xlabel("Observation number")
    ax.set_ylabel("Measured value")
plt.tight_layout()
plt.show()

**Interpretation:** A line implies an ordered progression or connection. A scatter plot treats the measurements as independent pairs. For the Cartwheel participants, a scatter plot is usually the appropriate choice.

<a id="custom"></a>
## 4. Customization: turn a default graph into an explanation

We can add marker shapes, transparency (`alpha`), tick labels, a highlighted dot, explanatory text, and a reference line. The highlighted point below is **a teaching example**, not a statistical outlier claim.

In [ ]:
demo = viz[["Wingspan", "CWDistance"]].dropna()
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(demo["Wingspan"], demo["CWDistance"], s=65,
           alpha=0.68, color="#26769b", edgecolor="white", linewidth=0.6,
           label="Participants")

# Highlight an observed point: use a real data row, not invented coordinates
point = demo.iloc[0]
ax.scatter(point["Wingspan"], point["CWDistance"],
           s=180, marker="*", color="crimson", zorder=5, label="Example point")
ax.annotate("This observed point", xy=(point["Wingspan"], point["CWDistance"]),
            xytext=(12, 18), textcoords="offset points",
            arrowprops={"arrowstyle": "->", "color": "crimson"},
            fontsize=10)
ax.axhline(demo["CWDistance"].median(), linestyle="--", color="gray",
           label="Median cartwheel distance")
ax.set(title="Wingspan and cartwheel distance", xlabel="Wingspan (dataset units)",
       ylabel="Cartwheel distance (dataset units)")
ax.tick_params(axis="both", labelsize=10)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

**Why “dataset units”?** The original lesson does not establish the units in a reliable metadata dictionary. Replace these with explicit units only after checking the source documentation. Avoid assigning “cm” or “inches” by assumption.

**Discussion:** Why does `alpha=0.68` help when points overlap? When would an annotation be distracting?

<a id="libraries"></a>
## 5. One dataset, three plotting libraries

- **Pandas**: quickest when data is already in a DataFrame.
- **Matplotlib**: most control over figures, axes, labels, styles, and placement.
- **Seaborn**: concise statistical visualizations built on Matplotlib.

All three are used below on the **same data**, so differences are about the plotting API—not changing observations.

In [ ]:
# Library 1: Pandas — convenient DataFrame plotting
ax = demo.plot.scatter(x="Wingspan", y="CWDistance", figsize=(7, 4.5),
                       color="teal", alpha=0.7, title="Pandas scatter plot")
ax.set_xlabel("Wingspan (dataset units)")
ax.set_ylabel("Cartwheel distance (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
# Library 2: Matplotlib — explicit control
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(demo["Wingspan"], demo["CWDistance"], color="teal", alpha=0.7)
ax.set(title="Matplotlib scatter plot", xlabel="Wingspan (dataset units)",
       ylabel="Cartwheel distance (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
# Library 3: Seaborn — statistical plots from named DataFrame columns
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(data=demo, x="Wingspan", y="CWDistance", color="teal", ax=ax)
ax.set(title="Seaborn scatter plot", xlabel="Wingspan (dataset units)",
       ylabel="Cartwheel distance (dataset units)")
plt.tight_layout()
plt.show()

### Side-by-side graphs: `plt.subplots(1, 2)`

`1, 2` means **one row, two columns**. Use `axes[0]` and `axes[1]` to target each panel. `sharey=True` gives panels the same y-axis scale, helping fair comparisons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

sns.scatterplot(data=viz, x="Wingspan", y="CWDistance", ax=axes[0],
                color="#217b95", s=60)
axes[0].set_title("Wingspan vs cartwheel distance")
axes[0].set_xlabel("Wingspan (dataset units)")
axes[0].set_ylabel("Cartwheel distance (dataset units)")

sns.scatterplot(data=viz, x="Height", y="CWDistance", ax=axes[1],
                color="#db7a30", s=60)
axes[1].set_title("Height vs cartwheel distance")
axes[1].set_xlabel("Height (dataset units)")
axes[1].set_ylabel("Cartwheel distance (dataset units)")

fig.suptitle("Two explanations worth comparing", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Three side-by-side plots: one row and three columns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bins in zip(axes, [5, 10, 20]):
    sns.histplot(data=viz, x="CWDistance", bins=bins, ax=ax,
                 color="#357b9d", edgecolor="white")
    ax.set_title(f"{bins} bins")
    ax.set_xlabel("Cartwheel distance")
    ax.set_ylabel("Count")
fig.suptitle("The same data can look different under different bin choices",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

<a id="histograms"></a>
## 6. Histogram: what does the distribution look like?

A histogram divides a numeric variable into **bins** and counts observations in each range. Changing the bins changes the visual appearance—not the underlying data. `stat="density"` changes the y-axis to probability *density* (the total bar **area** is 1; individual bar heights need not be below 1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(data=viz, x="CWDistance", bins=10, color="#2a7998",
             edgecolor="white", ax=axes[0])
axes[0].set(title="Frequency histogram", xlabel="Cartwheel distance", ylabel="Count")
sns.histplot(data=viz, x="CWDistance", bins=10, stat="density",
             color="#2a7998", alpha=0.55, ax=axes[1])
sns.kdeplot(data=viz, x="CWDistance", color="#b33d45", ax=axes[1])
axes[1].set(title="Density + KDE estimate", xlabel="Cartwheel distance", ylabel="Density")
plt.tight_layout()
plt.show()

**Interpretation checklist:** Where is the center? Is the distribution skewed? Are there gaps or extreme values? Does the appearance change when you choose five versus twenty bins? A KDE is a **smoothed estimate**, not the exact observed frequency distribution.

<a id="scatter"></a>
## 7. Scatter plot: do two measurements vary together?

A scatter plot visualizes paired numeric observations. A fitted line summarizes an overall trend but does **not** prove causality. `sns.regplot` adds a fitted regression line; use `scatter=False` to avoid drawing the points twice.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=viz, x="Wingspan", y="CWDistance", hue="Gender",
                s=75, alpha=0.85, ax=ax)
sns.regplot(data=viz, x="Wingspan", y="CWDistance", scatter=False,
            ci=None, color="black", line_kws={"linestyle": "--"}, ax=ax)
ax.set(title="Are wingspan and cartwheel distance associated?",
       xlabel="Wingspan (dataset units)", ylabel="Cartwheel distance (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
pairs = viz[["Wingspan", "CWDistance"]].dropna()
r = pairs["Wingspan"].corr(pairs["CWDistance"])
print(f"Pearson correlation r = {r:.3f} (n = {len(pairs)} complete pairs)")
print("Interpretation: sign indicates direction, magnitude indicates linear association;")
print("neither correlation nor a fitted regression line demonstrates causation.")

<a id="groups"></a>
## 8. Boxplot and violin plot: compare numeric distributions

A **boxplot** shows the median, interquartile range (IQR), whiskers, and possible outliers (by default, the 1.5×IQR rule). A **violin plot** displays a smoothed estimate of the distribution; it can be unstable for very small groups.

**Do not compare apples to oranges:** putting `Age`, `Height`, `Score`, and `CWDistance` on one shared y-axis can obscure their different units and scales. We compare **one variable** across categories instead.

In [ ]:
group_data = viz.dropna(subset=["Gender", "Height"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
sns.boxplot(data=group_data, x="Gender", y="Height", ax=axes[0],
            color="#87b8d3")
sns.stripplot(data=group_data, x="Gender", y="Height", ax=axes[0],
              color="#263b58", alpha=.65, jitter=.14, size=4)
axes[0].set(title="Boxplot + individual observations", xlabel="Gender", ylabel="Height (dataset units)")
sns.violinplot(data=group_data, x="Gender", y="Height", ax=axes[1],
               color="#98b9b1", inner="quartile", cut=0)
axes[1].set(title="Violin: estimated distribution shape", xlabel="Gender", ylabel="Height (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
group_summary = (group_data.groupby("Gender", observed=True)["Height"]
                 .agg(n="count", mean="mean", median="median", sd="std")
                 .round(2))
display(group_summary)
print("Compare medians, spread, sample sizes and overlap before stating a difference.")

<a id="counts"></a>
## 9. Count plot: how many observations are in each category?

Use a **count plot** for categorical variables, not a histogram. Direct labels help readers see sample sizes. If the categories have no inherent order, be consistent about their displayed order.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(data=viz, x="Gender", color="#4a90a4", ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, padding=3)
ax.set(title="Number of participants in each recorded group",
       xlabel="Gender (dataset categories)", ylabel="Participants")
ax.set_ylim(0, max(viz["Gender"].value_counts().max() * 1.18, 1))
plt.tight_layout()
plt.show()

<a id="research"></a>
## 10. Research lab: ask → plot → calculate → observe

These investigations **use the same Cartwheel dataset as the original notebook**, not simulated results. Each has (A) a question, (B) one or two figures, (C) a calculated numerical answer, and (D) a prompt for a defensible observation. Output values are calculated at runtime so the answers match the downloaded CSV.

**Important:** These are descriptive, exploratory classroom questions—not causal tests or evidence of population-level differences.

### Investigation A — Is wingspan related to cartwheel distance?

**Prediction:** Will the points slope upward, downward, or show no clear pattern?

**Visual evidence:** scatter plot plus a regression line. **Numeric evidence:** Pearson correlation.

In [ ]:
a = viz[["Wingspan", "CWDistance"]].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=a, x="Wingspan", y="CWDistance", ax=axes[0],
                color="#2a809c", s=75)
axes[0].set_title("Raw observations")
sns.regplot(data=a, x="Wingspan", y="CWDistance", ci=None, ax=axes[1],
            color="#a54b51", scatter_kws={"alpha": .7})
axes[1].set_title("Observed trend (descriptive fit)")
for ax in axes:
    ax.set_xlabel("Wingspan")
    ax.set_ylabel("Cartwheel distance")
plt.tight_layout(); plt.show()

r_a = a["Wingspan"].corr(a["CWDistance"])
direction = "positive" if r_a > 0 else "negative" if r_a < 0 else "approximately zero"
print(f"ANSWER: n={len(a)}, Pearson r={r_a:.3f}; direction: {direction}.")
print("OBSERVATION: Describe the overall trend, scatter around the line, and unusual points.")
print("LIMITATION: An association does not establish a causal effect.")

### Investigation B — Do recorded gender groups differ in height?

**Visual evidence:** boxplot + individual dots and violin plot. **Numerical evidence:** sample sizes and group medians. Compare the *observed sample*, not “all men” or “all women.”

In [ ]:
b = viz.dropna(subset=["Gender", "Height"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
sns.boxplot(data=b, x="Gender", y="Height", color="#89b8d2", ax=axes[0])
sns.stripplot(data=b, x="Gender", y="Height", color="black", alpha=.55,
              jitter=.15, ax=axes[0])
axes[0].set_title("Height: center, spread and actual points")
sns.violinplot(data=b, x="Gender", y="Height", color="#9dc2b4",
               cut=0, inner="quartile", ax=axes[1])
axes[1].set_title("Height: smoothed distribution estimate")
for ax in axes:
    ax.set_xlabel("Recorded gender category")
    ax.set_ylabel("Height (dataset units)")
plt.tight_layout(); plt.show()
summary_b = b.groupby("Gender")["Height"].agg(n="count", median="median", mean="mean").round(2)
display(summary_b)
print("ANSWER: The table reports each observed group's n, median and mean.")
print("OBSERVATION: Compare medians and overlap; mention unequal group sizes if present.")
print("LIMITATION: Small-group violin shapes can be unreliable; this is not a causal claim.")

### Investigation C — Are cartwheel distances concentrated in a narrow range?

**Visual evidence:** histogram with a median reference line and alternative binning. **Numeric evidence:** median and IQR.

In [ ]:
c = viz["CWDistance"].dropna()
median_c = c.median()
q1, q3 = c.quantile([.25, .75])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, nbins in zip(axes, [6, 14]):
    sns.histplot(c, bins=nbins, ax=ax, color="#407d9c", edgecolor="white")
    ax.axvline(median_c, color="crimson", linestyle="--", label=f"Median = {median_c:.1f}")
    ax.set(title=f"Cartwheel distance: {nbins} bins", xlabel="Distance", ylabel="Count")
    ax.legend()
plt.tight_layout(); plt.show()
print(f"ANSWER: n={len(c)}, median={median_c:.2f}, Q1={q1:.2f}, Q3={q3:.2f}, IQR={q3-q1:.2f}.")
print("OBSERVATION: Compare the histogram shapes; the data and summary statistics did not change.")

### Investigation D — Which recorded category is more common, and could unequal group sizes influence visual interpretation?

**Visual evidence:** count plot and height boxplot. **Numeric evidence:** counts and group sizes.

In [ ]:
d = viz.dropna(subset=["Gender"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=d, x="Gender", color="#3d8aa5", ax=axes[0])
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)
axes[0].set(title="Sample composition", xlabel="Recorded gender category", ylabel="Count")
sns.boxplot(data=d, x="Gender", y="Height", color="#e1a272", ax=axes[1])
axes[1].set(title="Height comparisons use unequal n if groups differ", xlabel="Gender", ylabel="Height")
plt.tight_layout(); plt.show()
counts_d = d["Gender"].value_counts()
print("ANSWER — observed group sizes:")
print(counts_d.to_string())
print("OBSERVATION: State whether the dataset is balanced across groups before interpreting a between-group plot.")

### A reproducible research conclusion template

> **Question:** We investigated whether __________.  
> **Figure(s):** Figure __ shows __________.  
> **Numeric evidence:** The sample contained __ observations; the relevant statistic was __.  
> **Observation:** In this sample, __________.  
> **Limitation:** The graph does not prove __________, and results may not generalize.

**Pair task:** Choose any two investigations and write a three-sentence Results + Observations paragraph. Avoid the words “causes,” “proves,” and “significant” unless the study design and analysis support them.

<a id="quality"></a>
## 11. Quality checklist and export

Before showing a graph in a report or presentation:

- **Match chart to variable types.** Numerical: histogram/scatter; categorical: count; numeric by category: box/violin.
- **Write informative labels and units.** Do not invent units absent from the source.
- **Avoid misleading axes.** Bar plots normally start at zero; scatter plots can use meaningful zoomed axes with transparency about scale.
- **Show sample size and uncertainty when relevant.** A smoothed violin from a tiny group may mislead.
- **Avoid redundant decoration.** Explain one insight, not every pixel.
- **Do not confuse correlation with causation.**
- **Save a high-resolution PNG or vector PDF.**

In [ ]:
# Production-ready, exportable example
fig, ax = plt.subplots(figsize=(8.5, 5))
sns.scatterplot(data=viz, x="Wingspan", y="CWDistance", hue="Gender",
                s=75, alpha=.8, ax=ax)
ax.set(title="Cartwheel distance versus wingspan",
       xlabel="Wingspan (dataset units)",
       ylabel="Cartwheel distance (dataset units)")
ax.legend(title="Recorded gender")
plt.tight_layout()
fig.savefig("cartwheel_scatter.png", dpi=250)
fig.savefig("cartwheel_scatter.pdf")  # vector format, where supported
plt.show()
print("Saved cartwheel_scatter.png and cartwheel_scatter.pdf")

<a id="challenge"></a>
## 12. In-class challenge — a two-panel research figure

**Research question:** Does `Height` or `Wingspan` appear more strongly associated with `CWDistance` in this sample?

1. Make two side-by-side scatterplots, keeping the same y-axis scale.
2. Add appropriate labels and a shared figure title.
3. Compute both Pearson correlations using complete pairs **for each comparison**.
4. Write a two-sentence observation and one limitation.

**Optional bridge to Lesson 041:** Using a MultiIndex dataset, select a numeric column with `.xs()` and plot its distribution. **Optional bridge to Lesson 042:** Merge a lookup table containing human-readable category labels, then use these labels in a count plot.

In [ ]:
# STUDENT STARTER — change/extend this to complete the challenge
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, predictor in zip(axes, ["Height", "Wingspan"]):
    sns.scatterplot(data=viz, x=predictor, y="CWDistance", ax=ax, s=65)
    ax.set_title(f"{predictor} vs cartwheel distance")
    ax.set_xlabel(f"{predictor} (dataset units)")
    ax.set_ylabel("Cartwheel distance (dataset units)")

fig.suptitle("Which measurement has the stronger linear association?", fontweight="bold")
plt.tight_layout()
plt.show()

for predictor in ["Height", "Wingspan"]:
    subset = viz[[predictor, "CWDistance"]].dropna()
    print(f"{predictor:9s}: n={len(subset):2d}, Pearson r={subset[predictor].corr(subset['CWDistance']):+.3f}")

### Instructor answer guide

- Compare the **absolute values** of the two correlations to discuss linear association strength, and the **signs** to discuss direction. Students should read the actual computed values, rather than asserting a predetermined winner.
- More strongly correlated in this sample does **not** mean the variable is the better causal explanation.
- If the two correlations use different subsets owing to missing values, mention the different sample sizes.
- A graph should clearly communicate the question, axes, observation, and limitation.

### Exit ticket

1. Which command changes the x-axis label?  
2. How do you add a red point at `(2, 3)`?  
3. What does `plt.subplots(1, 2)` mean?  
4. When would you choose a boxplot over a count plot?  
5. Why does a scatterplot not prove causation?

**Answers:** `ax.set_xlabel(...)`; `ax.scatter(2, 3, color="red")`; one row/two panels; numeric outcome compared across categories versus categorical frequencies; association can arise from other factors or chance.

---
**Sources and continuity:** Adapted and reorganized from your Lesson 043 notebook, reusing its Cartwheel CSV source. Lessons 041 and 042 are referenced as optional practice using selection and joins. Plot commands follow standard Pandas, Matplotlib, and Seaborn APIs.